In [1]:
# Cell 1: Import required libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score

In [20]:
# Cell 2: Load dataset
df = pd.read_csv("../../dataset/loan_approval_dataset.csv")
df.head()

,loan_id,no_of_dependents,education,self_employed,income_annum,loan_amount,loan_term,cibil_score,residential_assets_value,commercial_assets_value,luxury_assets_value,bank_asset_value,loan_status
0,1,2,Graduate,No,9600000,29900000,12,778,2400000,17600000,22700000,8000000,Approved
1,2,0,Not Graduate,Yes,4100000,12200000,8,417,2700000,2200000,8800000,3300000,Rejected
2,3,3,Graduate,No,9100000,29700000,20,506,7100000,4500000,33300000,12800000,Rejected
3,4,3,Graduate,No,8200000,30700000,8,467,18200000,3300000,23300000,7900000,Rejected
4,5,5,Not Graduate,Yes,9800000,24200000,20,382,12400000,8200000,29400000,5000000,Rejected


In [21]:
# Remove leading/trailing spaces from column names
df.columns = df.columns.str.strip()
# Print all column names to verify
print("Columns in dataset:\n", df.columns.tolist())

Columns in dataset:
 ['loan_id', 'no_of_dependents', 'education', 'self_employed', 'income_annum', 'loan_amount', 'loan_term', 'cibil_score', 'residential_assets_value', 'commercial_assets_value', 'luxury_assets_value', 'bank_asset_value', 'loan_status']


In [23]:
# Cell 3: Convert ApplicantIncome into categorical ranges
q1 = df["income_annum"].quantile(0.25)
q3 = df["income_annum"].quantile(0.75)

def income_category(income):
    if income <= q1:
        return "Low"
    elif income <= q3:
        return "Medium"
    else:
        return "High"

df["Income_Group"] = df["income_annum"].apply(income_category)
df[["income_annum", "Income_Group"]].head()

,income_annum,Income_Group
0,9600000,High
1,4100000,Medium
2,9100000,High
3,8200000,High
4,9800000,High


In [26]:
# Cell 4: Create Credit_History from cibil_score
# (Good if score >= 600)
df["Credit_History"] = df["cibil_score"].apply(lambda x: "Good" if x >= 600 else "Bad")
df[["cibil_score", "Credit_History"]].head()

,cibil_score,Credit_History
0,778,Good
1,417,Bad
2,506,Bad
3,467,Bad
4,382,Bad


In [34]:
# Cell 5: Use residential_assets_value as Property_Area substitute
df["Property_Area"] = df["residential_assets_value"]

# Cell 6: Convert residential_assets_value into categories
q1_asset = df["residential_assets_value"].quantile(0.25)
q3_asset = df["residential_assets_value"].quantile(0.75)

def property_category(value):
    if value <= q1_asset:
        return "Rural"      # low assets
    elif value <= q3_asset:
        return "Semiurban"  # medium assets
    else:
        return "Urban"      # high assets

df["Property_Area"] = df["residential_assets_value"].apply(property_category)
df[["residential_assets_value", "Property_Area"]].head()

,residential_assets_value,Property_Area
0,2400000,Semiurban
1,2700000,Semiurban
2,7100000,Semiurban
3,18200000,Urban
4,12400000,Urban


In [35]:
# Cell 6: Select required features
data = df[["Credit_History", "Property_Area", "Income_Group", "loan_status"]].copy()
data.dropna(inplace=True)
data.head()

,Credit_History,Property_Area,Income_Group,loan_status
0,Good,Semiurban,High,Approved
1,Bad,Semiurban,Medium,Rejected
2,Bad,Semiurban,High,Rejected
3,Bad,Urban,High,Rejected
4,Bad,Urban,High,Rejected


In [36]:
# Cell 7: Encode categorical variables
le_dict = {}

for col in data.columns:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    le_dict[col] = le

In [38]:
# Cell 8: Split dataset
X = data.drop("loan_status", axis=1)
y = data["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [39]:
# Cell 9: Train Naive Bayes classifier
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

GaussianNB()

In [41]:
# Cell 10: Predict for given applicant
# Credit_History = Good
# Property_Area = Urban (mapped → Graduate here)
# Income_Group = Medium
sample = pd.DataFrame({
    "Credit_History": [le_dict["Credit_History"].transform(["Good"])[0]],
    "Property_Area": [le_dict["Property_Area"].transform(["Urban"])[0]],
    "Income_Group": [le_dict["Income_Group"].transform(["Medium"])[0]]
})

prediction = nb_model.predict(sample)
result = le_dict["loan_status"].inverse_transform(prediction)
print("Prediction (Naive Bayes):", result[0])

Prediction (Naive Bayes):  Approved


In [42]:
# Cell 11: Evaluate Naive Bayes
y_pred_nb = nb_model.predict(X_test)
print("Naive Bayes Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall:", recall_score(y_test, y_pred_nb))

Naive Bayes Performance:
Accuracy: 0.8536299765807962
Precision: 0.7188208616780045
Recall: 0.9968553459119497


In [43]:
# Cell 12: Logistic Regression
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print("Logistic Regression Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall:", recall_score(y_test, y_pred_lr))

Logistic Regression Performance:
Accuracy: 0.8536299765807962
Precision: 0.7188208616780045
Recall: 0.9968553459119497


In [44]:
# Cell 13: KNN Classifier
knn_model = KNeighborsClassifier(n_neighbors=5)
knn_model.fit(X_train, y_train)
y_pred_knn = knn_model.predict(X_test)

print("KNN Performance:")
print("Accuracy:", accuracy_score(y_test, y_pred_knn))
print("Precision:", precision_score(y_test, y_pred_knn))
print("Recall:", recall_score(y_test, y_pred_knn))

KNN Performance:
Accuracy: 0.797423887587822
Precision: 0.7244582043343654
Recall: 0.7358490566037735


In [45]:
# Cell 14: Compare all models
results = pd.DataFrame({
    "Model": ["Naive Bayes", "Logistic Regression", "KNN"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_knn)
    ],
    "Precision": [
        precision_score(y_test, y_pred_nb),
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_knn)
    ],
    "Recall": [
        recall_score(y_test, y_pred_nb),
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_knn)
    ]
})

results

,Model,Accuracy,Precision,Recall
0,Naive Bayes,0.853630,0.718821,0.996855
1,Logistic Regression,0.853630,0.718821,0.996855
2,KNN,0.797424,0.724458,0.735849
